In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from fnmatch import fnmatch
import plotly.express as px 
import statsmodels.api as sm
import statsmodels.formula.api as smf 
from patsy import dmatrices

In [2]:

root = (Path.cwd() / "apt_prices_poland")
if not root.exists():
    root = Path.cwd().parent / "apt_prices_poland"  

skip_pattern = "apartments_rent_pl_*.csv"

csv_paths = [
    path
    for path in root.rglob("*.csv")
    if not fnmatch(path.name, skip_pattern)
]

if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under {root}")

frames = [pd.read_csv(p) for p in csv_paths]
combined = pd.concat(frames, ignore_index=True)

In [3]:
combined

,id,city,type,squareMeters,rooms,floor,floorCount,buildYear,latitude,longitude,...,pharmacyDistance,ownership,buildingMaterial,condition,hasParkingSpace,hasBalcony,hasElevator,hasSecurity,hasStorageRoom,price
0,a01d82c9529f98a54d64b9e061c9a73b,szczecin,apartmentBuilding,105.00,4.0,3.0,4.0,2016.0,53.431503,14.485820,...,0.335,condominium,brick,premium,no,yes,yes,no,no,1199999
1,d68ea84e5d35da9e282150332bdc22f3,szczecin,blockOfFlats,94.40,4.0,4.0,5.0,NaN,53.441253,14.511030,...,0.051,condominium,brick,premium,yes,yes,no,no,yes,1150000
2,420295cc23d693fdffd5ccc9ba35ba98,szczecin,NaN,48.29,2.0,8.0,11.0,2014.0,53.399444,14.526111,...,0.141,condominium,NaN,NaN,yes,yes,yes,no,yes,625000
3,7d0c31d5409caab173571cce3dcdf702,szczecin,blockOfFlats,68.61,3.0,4.0,4.0,1997.0,53.456213,14.583222,...,0.304,condominium,brick,NaN,no,yes,no,no,yes,550000
4,7ec72a2301d950ae17926c3c1e67a0ed,szczecin,tenement,35.92,2.0,NaN,NaN,NaN,53.424203,14.543550,...,0.329,condominium,brick,low,yes,no,no,no,no,299000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
195563,5803d8594e9d69c558ae99bb07ccaca0,bydgoszcz,tenement,106.00,5.0,2.0,2.0,NaN,53.121506,17.994189,...,0.396,condominium,brick,NaN,no,yes,no,no,no,742000
195564,204c93689b7cdc62a17ef3c0dbf7034a,bydgoszcz,tenement,98.00,3.0,NaN,3.0,1925.0,53.124655,18.008459,...,0.273,condominium,brick,NaN,no,no,no,no,yes,499000
195565,bb19da639a2de8bba49be2ca49053c87,bydgoszcz,tenement,108.96,5.0,2.0,4.0,1889.0,53.131748,18.000648,...,0.143,condominium,brick,NaN,no,no,no,no,yes,795000
195566,1e7f4f1fdfea31eb84e071d697839632,bydgoszcz,NaN,50.12,2.0,1.0,1.0,NaN,53.129657,18.003888,...,0.250,condominium,brick,NaN,yes,no,no,no,no,360000


In [6]:
df = combined.copy()

---

---

In [7]:
df = df.copy().reset_index(drop=True)
df.columns = df.columns.str.strip()

rename_map = {
    'centreDistance': 'centre_distance',
    'poiCount': 'poi_count',
    'schoolDistance': 'school_distance',
    'clinicDistance': 'clinic_distance',
    'postOfficeDistance': 'post_office_distance',
    'kindergartenDistance': 'kindergarten_distance',
    'restaurantDistance': 'restaurant_distance',
    'collegeDistance': 'college_distance',
    'pharmacyDistance': 'pharmacy_distance',
    'squareMeters': 'size_sqm',
    'buildYear': 'build_year',
    'floorCount': 'floor_count',
}
df.rename(columns={k:v for k,v in rename_map.items() if k in df.columns}, inplace=True)

In [ ]:
# Build composite amenity distance (km) and outcome
amen_cols = [c for c in [
    'school_distance','clinic_distance','post_office_distance',
    'kindergarten_distance','restaurant_distance','college_distance','pharmacy_distance'
] if c in df.columns]
if not amen_cols:
    raise KeyError("No amenity distance columns found to create 'amenity_distance_km'.")

df['amenity_distance_km'] = df[amen_cols].min(axis=1)

# Keep only positive price and create ln_price
df = df[pd.to_numeric(df['price'], errors='coerce') > 0].copy()
df['ln_price'] = np.log(df['price'])

In [ ]:
# Controls and RHS (only real columns)
controls = [c for c in ['size_sqm','rooms','floor','floor_count','build_year'] if c in df.columns]
rhs_vars = ['centre_distance','poi_count','amenity_distance_km'] + controls

# C(city) in the formula; keep raw 'city' for clustering
if 'city' not in df.columns:
    raise KeyError("Column 'city' is required for C(city) and clustering.")
df['city'] = df['city'].astype(str).str.strip()

formula = 'ln_price ~ ' + ' + '.join(rhs_vars) + ' + C(city)'
print('Formula:', formula)

Formula: ln_price ~ centre_distance + poi_count + amenity_distance_km + size_sqm + rooms + floor + floor_count + build_year + C(city)


In [ ]:
# Build design matrices(drops NA consistently)
y, X = dmatrices(formula, data=df, return_type='dataframe')
# Align cluster groups exactly to y/X rows
groups = df.loc[y.index, 'city'].astype(str).str.strip().fillna('_NA_')

In [11]:
# 5) Fit OLS and apply robust covariance
res = sm.OLS(y, X).fit()
if groups.nunique() >= 2:
    res = res.get_robustcov_results(cov_type='cluster', groups=groups.values, use_correction=True)
else:
    res = res.get_robustcov_results(cov_type='HC1')

print(res.summary().tables[1])

                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  6.3429      0.595     10.660      0.000       5.067       7.619
C(city)[T.bydgoszcz]      -0.1099      0.007    -16.874      0.000      -0.124      -0.096
C(city)[T.czestochowa]    -0.2459      0.004    -60.125      0.000      -0.255      -0.237
C(city)[T.gdansk]          0.5197      0.015     34.830      0.000       0.488       0.552
C(city)[T.gdynia]          0.4261      0.017     25.400      0.000       0.390       0.462
C(city)[T.katowice]        0.0246      0.009      2.743      0.016       0.005       0.044
C(city)[T.krakow]          0.5857      0.011     53.388      0.000       0.562       0.609
C(city)[T.lodz]           -0.0246      0.009     -2.711      0.017      -0.044      -0.005
C(city)[T.lublin]          0.0738      0.003     21.800      0.000       0.067       0.081

/Users/heesung/Documents/Poland/AGH/2025_Winter/Diploma/Raw/.venv/lib/python3.12/site-packages/statsmodels/base/model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 22, but rank is 8
  warnings.warn('covariance of constraints does not have full '


In [21]:
print(res.summary())

                            OLS Regression Results                            
Dep. Variable:               ln_price   R-squared:                       0.806
Model:                            OLS   Adj. R-squared:                  0.806
Method:                 Least Squares   F-statistic:                     7623.
Date:                Thu, 06 Nov 2025   Prob (F-statistic):           4.02e-24
Time:                        20:38:55   Log-Likelihood:                 23029.
No. Observations:              135029   AIC:                        -4.601e+04
Df Residuals:                  135006   BIC:                        -4.579e+04
Df Model:                          22                                         
Covariance Type:              cluster                                         
                             coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------
Intercept                  6

In [ ]:
# Convert betas to scenario coefficients + 95% CIs
names = list(X.columns)
b = pd.Series(np.asarray(res.params), index=names)
ci_arr = np.asarray(res.conf_int())
ci = pd.DataFrame(ci_arr, index=names, columns=['ci_lo','ci_hi'])  

def ci_pair(name):
    if name not in ci.index:
        return (np.nan, np.nan)
    return float(ci.loc[name,'ci_lo']), float(ci.loc[name,'ci_hi'])

# Distance effects are negative in logs; your service needs magnitudes for penalties
COEF_CENTER_PER_KM = -float(b.get('centre_distance', np.nan))                 # per km farther/closer
COEF_POI_PER_UNIT  =  float(b.get('poi_count', np.nan))                       # per additional POI
COEF_AMENITY_PER_M = -float(b.get('amenity_distance_km', np.nan)) / 1000.0    # km -> per meter

center_ci    = tuple(-x for x in ci_pair('centre_distance'))
poi_ci       = ci_pair('poi_count')
amen_ci_km   = ci_pair('amenity_distance_km')
amen_ci_m    = (-(amen_ci_km[0]/1000.0), -(amen_ci_km[1]/1000.0)) if np.isfinite(amen_ci_km[0]) and np.isfinite(amen_ci_km[1]) else (np.nan, np.nan)

print('\n--- Scenario coefficients (data-backed) ---')
print(f'COEF_CENTER_PER_KM = {COEF_CENTER_PER_KM:.6f}   CI: {center_ci}')
print(f'COEF_POI_PER_UNIT  = {COEF_POI_PER_UNIT:.6f}   CI: {poi_ci}')
print(f'COEF_AMENITY_PER_M = {COEF_AMENITY_PER_M:.8f}   CI: {amen_ci_m}')



--- Scenario coefficients (data-backed) ---
COEF_CENTER_PER_KM = 0.030159   CI: (0.04211941732298218, 0.018198523568437333)
COEF_POI_PER_UNIT  = 0.001901   CI: (0.001228852609446192, 0.002572782542648199)
COEF_AMENITY_PER_M = 0.00002358   CI: (6.230395162608525e-05, -1.5148953990515277e-05)


In [20]:
# Optional transit coefficients if present in your df
if 'transit_score' in df.columns:
    # Extend formula with transit_score, rebuild y/X to avoid NA drift
    tf = formula + ' + transit_score'
    y2, X2 = dmatrices(tf, data=df, return_type='dataframe')
    g2 = df.loc[y2.index, 'city'].astype(str).str.strip().fillna('_NA_')
    res2 = sm.OLS(y2, X2).fit()
    if g2.nunique() >= 2:
        res2 = res2.get_robustcov_results(cov_type='cluster', groups=g2.values, use_correction=True)
    else:
        res2 = res2.get_robustcov_results(cov_type='HC1')
    beta_ts = float(res2.params.get('transit_score', np.nan))
    ts_ci   = (float(res2.conf_int().loc['transit_score',0]), float(res2.conf_int().loc['transit_score',1])) if 'transit_score' in res2.params.index else (np.nan, np.nan)
    print(f'COEF_TRANSIT_PER_POINT = {beta_ts:.6f}  CI: {ts_ci}')
else:
    print('COEF_TRANSIT_PER_POINT: transit_score not found (skipped).')

print('\nNote: COEF_TRANSIT_UPGRADE requires a treated×post design (DiD). Provide those fields to estimate exp(beta)-1.')

COEF_TRANSIT_PER_POINT: transit_score not found (skipped).

Note: COEF_TRANSIT_UPGRADE requires a treated×post design (DiD). Provide those fields to estimate exp(beta)-1.
